In [ ]:
from pathlib import Path
import ixmp4
import pyam
import nomenclature

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/GENIE/").iterdir())
    ]
)

In [ ]:
df.rename(
    model={
        "MESSAGEix-GLOBIOM_1.1": "MESSAGEix-GLOBIOM 1.1",
    },
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i, 
                (
                    ("GENIE DACCS " + i[11:])
                )
            ) for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
#definition = nomenclature.DataStructureDefinition("../definitions/")
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
df.rename(
    region={
        "R5ASIA": "Asia (R5)",
        "R5LAM": "Latin America (R5)",
        "R5MAF": "Middle East & Africa (R5)",
        "R5OECD90+EU": "OECD & EU (R5)",
        "R5REF": "Reforming Economies (R5)",
        "R5ROWO": "Other (R5)",
    },
    inplace=True,
)

In [ ]:
df.filter(
    region=[
        "China & Centrally Planned Asia",
        "Developed Regions",
        "Latin America",
        "Middle East & Africa",
        "South & South East Asia",
    ],
    keep=False,
    inplace=True,
)

In [ ]:
definition.validate(df, dimensions=["region"])

In [ ]:
project = ["navigate", "engage"]
legacy_mapping = {}


for _project in project:
    for code, attrs in definition.variable.items():
        if _project in attrs.extra_attributes:
            legacy_mapping[attrs.__getattr__(_project)] = code
    
    df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "US$2010/GJ": "USD_2010/GJ",
        "bn m2": "billion m2",
        "bn tkm/yr": "billion tkm/yr",
        "bn pkm/yr": "billion pkm/yr",
    },
    inplace=True,
)

In [ ]:
# update carbon-management variables
variable_mapping = {
    "Agricultural Demand|Energy": "Agricultural Demand|Crops|Bioenergy",
    "Agricultural Demand|Energy|Crops|1st generation": "Agricultural Demand|Crops|Bioenergy|1st Generation",
    "Agricultural Demand|Energy|Crops|2nd generation": "Agricultural Demand|Crops|Bioenergy|2nd Generation",
    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Yield|Cereal": "Yield|Cropland|Cereals",
    "Yield|Oilcrops": "Yield|Cropland|Oil Crops",
    "Yield|Sugarcrops": "Yield|Cropland|Sugar Crops",
    "Final Energy|Greenhouse Gas Removal|Electricity|Direct Air Capture": "Final Energy|Carbon Management|Direct Air Capture|Electricity",
    "Final Energy|Greenhouse Gas Removal|Gases|Direct Air Capture": "Final Energy|Carbon Management|Direct Air Capture|Gases",
}

df.rename(variable=variable_mapping, inplace=True)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "Capacity Additions|Electricity|Storage Capacity",
        "Capacity|Electricity|Storage",
        "Cumulative Capacity*",
        "Secondary Energy",
        "Diagnostics|MAGICC6*",
        "Forcing*",
        "OM Cost*",
        "Additional Investment for DACCS",
        "Price|Agriculture*"
    ],
    keep=False,
    inplace=True
)

In [ ]:
definition.validate(df, dimensions=["variable"])

In [ ]:
df.filter(variable=definition.variable, inplace=True)

In [ ]:
validation_args = ["upper_bound", "lower_bound", "value", "rtol", "atol", "range"]

validation_list = list()

for name, variable in definition.variable.items():
    if any([i in validation_args for i in variable.extra_attributes]):
        validation_list.append(
            dict(
                variable=name,
                validation=[dict([(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])]
            )
        )

In [ ]:
validator = nomenclature.processor.DataValidator(criteria_items=validation_list, file=".")

In [ ]:
corrected_df = df.filter(model="COFFEE 1.5", variable="Carbon Capture|Industrial Processes")
corrected_df._data = - corrected_df._data

df = pyam.concat(
    [
        df.filter(model="COFFEE 1.5", variable="Carbon Capture|Industrial Processes", keep=False),
        corrected_df,
    ]
)

In [ ]:
# incorrectly aggregated variables
df.filter(
    variable=[
        "Terrestrial Biodiversity|Biodiversity Intactness Index",
        "Terrestrial Biodiversity|Mean Species Abundance|Plants",
        "Consumption|*",
        "Income|*",
        "Efficiency|*",
    ],
    region=[
        "World",
        "*(R5)",
        "*(R9)",
        "*(R10)",
    ],
    keep=False,
    inplace=True,
)

df.filter(
    model="IMAGE 3.3",
    variable=[
        "Terrestrial Biodiversity|Biodiversity Intactness Index",
    ],
    region=[
        "European Union and United Kingdom",
    ],
    keep=False,
    inplace=True,
)

df.filter(
    model="MESSAGEix-GLOBIOM 1.1-BMT-R12",
    variable=[
        "Efficiency*",
    ],
    region=[
        "European Union and United Kingdom",
    ],
    keep=False,
    inplace=True,
)

In [ ]:
validator.apply(df)

In [ ]:
validation_args

In [ ]:
definition.validate(df)

In [ ]:
df.set_meta("GENIE [European Research Council]", "Project")
df.set_meta("Gidden et al. (2023)", "Scientific Manuscript (Citation)")
df.set_meta("10.1088/1748-9326/acd8d5", "Scientific Manuscript (DOI)")

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
for model in df.model:
    df.filter(model=model).to_ixmp4(platform)
    print(model)

In [ ]:
platform.runs.tabulate(scenario="NAV*", default_only=False, is_default=False)

In [ ]:
df_remind = df.filter(model="REMIND-MAgPIE 3.3-4.8 - Integrated Physical Damages (median)")

In [ ]:
df_remind.region

In [ ]:
run.iamc.tabulate()

In [ ]:
run = platform.runs.get("MESSAGEix-GLOBIOM 2.0-M-R12-NGFS", "NGFS Phase 5-Current Policies", version=1)
_df = df.filter(model=run.model.name, scenario=run.scenario.name)
run.iamc.add(_df.data)
run.meta = dict(_df.meta.loc[(run.model.name, run.scenario.name)])
run.set_as_default()

In [ ]:
runs[0].scenario.name

In [ ]:
df_upload.model

In [ ]:
for model in ['WITCH 5.0']:
    _df = df_upload.filter(model=model)
    _df.to_ixmp4(platform)
    print("Finished model " + model)

In [ ]:
runs = platform.runs.list(scenario="NAV*", default_only=False, is_default=False)

In [ ]:
runs

In [ ]:
for run in runs:
    _df = df_upload.filter(model=run.model.name, scenario=run.scenario.name)
    run.iamc.add(_df.data)
    run.meta = dict(_df.meta.loc[(run.model.name, run.scenario.name)])
    run.set_as_default()

In [ ]:
for model in ['JRC-GEM-E3 v2021']:
    _df = df_upload.filter(model=model)
    for _, scenario in _df.index:
        try:
            platform.runs.get(model=model, scenario=scenario)
            print("Already exists: " + scenario)
        except:
            df_upload.filter(model=model, scenario=scenario).to_ixmp4(platform)
            print("Uploaded: " + scenario)

In [ ]:
len(df.index)

In [ ]:
runs = platform.runs.tabulate(scenario="NAVIGATE Sup*", default_only=False)

In [ ]:
runs.model.unique()

In [ ]:
df.model